# recs_009 - Offline eval pipeline vs frozen baseline parity

Purpose: validate that latest **offline eval** outputs match the frozen baseline within tolerance for baseline methods (**ranking contract** + **retrieval contract**).

## Prerequisites

1. Run job to generate artifacts:
   - `python scripts/recs_job_eval_retrieval.py configs/recs_job_eval_retrieval.json`
2. Ensure baseline JSON exists (dual snapshot):
   - `python scripts/recs_job_eval_retrieval.py configs/recs_job_eval_retrieval.json --write-baseline`
3. Confirm artifacts under `artifacts/recs/offline_eval/runs/latest/`:
   - `eval_ranking_overall.csv`
   - `eval_retrieval_overall.csv`
   - `eval_retrieval_baseline_overall.json`

This notebook compares:
- Ranking table vs `baseline["ranking_overall_by_method"]` (fallback: legacy `overall_by_method`)
- Retrieval table vs `baseline["retrieval_overall_by_method"]`

**Ranking** metrics checked: `Hit@K`, `Recall@K`, `MAP@K`, `NDCG@K`, `MRR`  
**Retrieval** metrics checked: `Hit@K`, `Precision@K`, `Recall@K`

**Methods:** `raw`, `popularity_train`, `multi_mean_train`

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd


def _find_repo_root(start: Path) -> Path:
    here = start.resolve()
    for d in [here, *here.parents]:
        if (d / "pyproject.toml").is_file():
            return d
    raise RuntimeError(f"Could not find repo root from start={start}")


# --- config ---
REPO_ROOT = _find_repo_root(Path.cwd())
EVAL_ROOT = REPO_ROOT / "artifacts" / "recs" / "offline_eval" / "runs" / "latest"
PIPELINE_RANKING_PATH = EVAL_ROOT / "eval_ranking_overall.csv"
PIPELINE_RETRIEVAL_PATH = EVAL_ROOT / "eval_retrieval_overall.csv"
BASELINE_JSON_PATH = EVAL_ROOT / "eval_retrieval_baseline_overall.json"

METHODS = ["raw", "popularity_train", "multi_mean_train"]
METRICS_RANKING = ["Hit@K", "Recall@K", "MAP@K", "NDCG@K", "MRR"]
METRICS_RETRIEVAL = ["Hit@K", "Precision@K", "Recall@K"]
TOLERANCE = 1e-3

In [2]:
if not PIPELINE_RANKING_PATH.is_file():
    raise FileNotFoundError(f"Missing ranking overall CSV: {PIPELINE_RANKING_PATH}")
if not PIPELINE_RETRIEVAL_PATH.is_file():
    raise FileNotFoundError(f"Missing retrieval overall CSV: {PIPELINE_RETRIEVAL_PATH}")
if not BASELINE_JSON_PATH.is_file():
    raise FileNotFoundError(
        f"Missing frozen baseline JSON: {BASELINE_JSON_PATH}\n"
        "Create it with:\n"
        "python scripts/recs_job_eval_retrieval.py "
        "configs/recs_job_eval_retrieval.json --write-baseline"
    )

ranking_df = pd.read_csv(PIPELINE_RANKING_PATH)
retr_df = pd.read_csv(PIPELINE_RETRIEVAL_PATH)
payload = json.loads(BASELINE_JSON_PATH.read_text(encoding="utf-8"))
rank_snap = payload.get("ranking_overall_by_method") or payload.get("overall_by_method", {})
retr_snap = payload.get("retrieval_overall_by_method", {})
if not rank_snap:
    raise ValueError(f"Baseline JSON missing ranking snapshot: {BASELINE_JSON_PATH}")
if not retr_snap:
    raise ValueError(
        f"Baseline JSON missing retrieval_overall_by_method. Re-freeze with --write-baseline: {BASELINE_JSON_PATH}"
    )

print("Loaded:")
print("-", PIPELINE_RANKING_PATH, "rows:", len(ranking_df))
print("-", PIPELINE_RETRIEVAL_PATH, "rows:", len(retr_df))
print("-", BASELINE_JSON_PATH)

Loaded:
- /home/ryanr/workspace/steam_recommendations/artifacts/recs/eval/eval_retrieval_overall.csv
- /home/ryanr/workspace/steam_recommendations/artifacts/recs/eval/eval_retrieval_baseline_overall.json
pipeline rows: 3 baseline rows: 3


In [3]:
def _snap_to_df(by_method: dict[str, dict[str, float]], metrics: list[str]) -> pd.DataFrame:
    rows = []
    for method, mvals in by_method.items():
        row = {"method": method}
        for metric in metrics:
            row[metric] = float(mvals[metric])
        rows.append(row)
    return pd.DataFrame(rows)


def parity_merge(
    pipeline_df: pd.DataFrame,
    by_method: dict[str, dict[str, float]],
    metrics: list[str],
    *,
    label: str,
) -> pd.DataFrame:
    required_cols = ["method", *metrics]
    reference_df = _snap_to_df(by_method, metrics)
    for c in required_cols:
        if c not in pipeline_df.columns:
            raise ValueError(f"[{label}] pipeline CSV missing column: {c}")
        if c not in reference_df.columns:
            raise ValueError(f"[{label}] baseline snapshot missing column: {c}")

    pipeline_sub = pipeline_df[pipeline_df["method"].isin(METHODS)][required_cols].copy()
    reference_sub = reference_df[reference_df["method"].isin(METHODS)][required_cols].copy()

    missing_methods_pipeline = sorted(set(METHODS) - set(pipeline_sub["method"].astype(str)))
    missing_methods_reference = sorted(set(METHODS) - set(reference_sub["method"].astype(str)))
    if missing_methods_pipeline:
        raise ValueError(f"[{label}] pipeline CSV missing methods: {missing_methods_pipeline}")
    if missing_methods_reference:
        raise ValueError(f"[{label}] baseline snapshot missing methods: {missing_methods_reference}")

    compare_df = pipeline_sub.merge(
        reference_sub,
        on="method",
        suffixes=("_pipeline", "_reference"),
        validate="one_to_one",
    )
    for metric in metrics:
        compare_df[f"{metric}_delta"] = compare_df[f"{metric}_pipeline"] - compare_df[f"{metric}_reference"]
        compare_df[f"{metric}_abs_delta"] = compare_df[f"{metric}_delta"].abs()

    abs_delta_cols = [f"{m}_abs_delta" for m in metrics]
    compare_df["max_abs_delta"] = compare_df[abs_delta_cols].max(axis=1)
    return compare_df

In [4]:
print("=== Ranking parity (eval_ranking_overall vs baseline) ===")
compare_ranking = parity_merge(ranking_df, rank_snap, METRICS_RANKING, label="ranking")
summary_r = ["method", "max_abs_delta"] + [f"{m}_abs_delta" for m in METRICS_RANKING]
display(compare_ranking[summary_r].sort_values("max_abs_delta", ascending=False))

print("=== Retrieval parity (eval_retrieval_overall vs baseline) ===")
compare_retrieval = parity_merge(retr_df, retr_snap, METRICS_RETRIEVAL, label="retrieval")
summary_t = ["method", "max_abs_delta"] + [f"{m}_abs_delta" for m in METRICS_RETRIEVAL]
display(compare_retrieval[summary_t].sort_values("max_abs_delta", ascending=False))

,method,max_abs_delta,Hit@K_abs_delta,Recall@K_abs_delta,MAP@K_abs_delta,NDCG@K_abs_delta,MRR_abs_delta
2,raw,9.020562e-17,0.0,4.163336e-17,1.387779e-17,7.632783e-17,9.020562e-17
1,multi_mean_train,8.326673e-17,0.0,1.387779e-17,7.285839e-17,2.775558e-17,8.326673e-17
0,popularity_train,6.938894e-17,0.0,2.775558e-17,6.938894e-18,1.387779e-17,6.938894e-17


In [5]:
def _collect_violations(compare_df: pd.DataFrame, metrics: list[str]) -> list[tuple[str, str, float]]:
    out: list[tuple[str, str, float]] = []
    for _, row in compare_df.iterrows():
        method = str(row["method"])
        for metric in metrics:
            abs_delta = float(row[f"{metric}_abs_delta"])
            if abs_delta > TOLERANCE:
                out.append((method, metric, abs_delta))
    return out


violations = _collect_violations(compare_ranking, METRICS_RANKING) + _collect_violations(
    compare_retrieval, METRICS_RETRIEVAL
)

if violations:
    msg = "\n".join([f"{m}.{metric}: abs_delta={d:.6f} > tol={TOLERANCE:.6f}" for m, metric, d in violations])
    raise AssertionError("Parity check failed:\n" + msg)

print(f"PASS: ranking + retrieval parity within tolerance={TOLERANCE:.6f}")

PASS: pipeline/reference parity within tolerance=0.001000
